# ScholarAI v3 SOTA: Llama-3 8B Backend
This notebook deploys the **Deep Semantic Evasion** engine. It uses **AMR Graph Parsing** to strip AI syntax and **Llama-3 8B** (pivoted from Gemma-4 for stability) to generate human-like academic text.

### ⚠️ Prerequisites
Before running, click the **Key icon (Secrets)** on the left and add:
1. `HF_TOKEN`: Your HuggingFace token (must have access to Llama-3).
2. `NGROK_TOKEN`: Your ngrok authentication token.
3. **Enable "Notebook access"** for both.

## Step 1: Environment & System Dependency Setup
This step installs the correct CUDA-aligned PyTorch stack and pins critical dependencies to avoid the `Gemma4Config`/`torchvision` import errors.

In [ ]:
# 1. Reset and Clone
%cd /content
!rm -rf sensorspine-humaniser-v3
!git clone https://github.com/NandishSinha1403/sensorspine-humaniser-v3.git

# 2. System dependencies
!apt-get install -y redis-server > /dev/null

# 3. Install CUDA 12.1 Aligned PyTorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 4. Install Project Dependencies (Avoiding overwriting torch/torchvision)
!pip install --upgrade amrlib fastapi uvicorn "pydantic<=2.12.3" accelerate bitsandbytes trl peft datasets python-multipart pyngrok penman unidecode huggingface_hub sentencepiece protobuf word2number celery redis python-jose[cryptography] passlib[bcrypt] slowapi "numpy<2.1" "pillow<12.0"
!pip install git+https://github.com/huggingface/transformers.git

# 5. Automated Authentication
from google.colab import userdata
from huggingface_hub import login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Successfully authenticated with HuggingFace.")
except Exception as e:
    print(f"❌ Authentication Error: {e}. Check your Colab Secrets for 'HF_TOKEN'.")

/content
Cloning into 'sensorspine-humaniser-v3'...
remote: Enumerating objects: 541, done.
remote: Counting objects: 100% (541/541), done.
remote: Compressing objects: 100% (408/408), done.
remote: Total 541 (delta 196), reused 470 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (541/541), 942.68 KiB | 16.25 MiB/s, done.
Resolving deltas: 100% (196/196), done.
Looking in indexes: https://download.pytorch.org/whl/cu121
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-beckrdk4
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-beckrdk4
  Resolved https://github.com/huggingface/transformers.git to commit f15fc1e8f8939cdd1d4e54c5cd409317c5a9e5ce
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Successfully authenticated with HuggingFace.


## Step 2: Deploy AMR Models
Downloads the Transformer-based models for semantic parsing and generation.

In [ ]:
%cd /content/sensorspine-humaniser-v3/backend
!mkdir -p models

print("Downloading AMR Parsing Model (STOG)...")
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xzf model_parse_xfm_bart_base-v0_1_0.tar.gz -C models/
!rm model_parse_xfm_bart_base-v0_1_0.tar.gz
!mv models/model_parse_xfm_bart_base-v0_1_0 models/model_stog

print("\nDownloading AMR Generation Model (GTOS)...")
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/model_generate_t5wtense-v0_1_0/model_generate_t5wtense-v0_1_0.tar.gz
!tar -xzf model_generate_t5wtense-v0_1_0.tar.gz -C models/
!rm model_generate_t5wtense-v0_1_0.tar.gz
!mv models/model_generate_t5wtense-v0_1_0 models/model_gtos

print("\n✅ AMR Pipeline Ready.")

/content/sensorspine-humaniser-v3/backend
model_parse_xfm_bar 100%[===================>] 492.07M   251MB/s    in 2.0s    

model_generate_t5wt 100%[===================>] 787.17M   141MB/s    in 5.8s    

✅ AMR Pipeline Ready.


## Step 3: Start Tunnel & Full Pipeline
This cell starts Redis, the Celery worker (Llama-3), and the FastAPI gateway via ngrok.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

try:
    # Set up ngrok tunnel
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)

    # Close existing tunnels
    ngrok.kill()

    # Connect to port 8000
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 BACKEND ACTIVE")
    print(f"COPY THIS URL TO FRONTEND: {public_url}")
    print("================================================\n")

    # Orchestrate Services
    !chmod +x start.sh
    !./start.sh
except Exception as e:
    print(f"❌ Failed to start pipeline: {e}. Check your Colab Secrets for 'NGROK_TOKEN'.")


🚀 BACKEND ACTIVE
COPY THIS URL TO FRONTEND: https://broodless-suzan-nonprohibitively.ngrok-free.dev

--------------------------------------------------------
🚀 Starting ScholarAI v3 SOTA Pipeline
--------------------------------------------------------
[1/3] Starting Redis Server...
✅ Redis is running.
[2/3] Starting Celery Worker (Gemma/AMR/Judge)...
[3/3] Starting FastAPI Gateway at http://0.0.0.0:8000
--------------------------------------------------------
/usr/local/lib/python3.12/dist-packages/celery/platforms.py:841: SecurityWarning: You're running the worker with superuser privileges: this is
absolutely not recommended!

Please specify a different user using the --uid option.

User information: uid=0 euid=0 gid=0 egid=0

  warnings.warn(SecurityWarning(ROOT_DISCOURAGED.format(
INFO:     Started server process [17386]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
 
 ----

In [ ]:
# FINAL PITCH-READY EMERGENCY RESET & LAUNCH
# Paste this into a single Colab cell to forcefully refresh and restart the service.

import os
from google.colab import userdata
from pyngrok import ngrok

# 1. Kill everything (Old workers, old redis, old tunnels)
print("🛑 Terminating old background processes...")
!pkill -9 -f celery
!pkill -9 -f uvicorn
!pkill -9 -f redis-server
ngrok.kill()

# 2. Pull the latest stability & debug code
print("\n🔄 Syncing latest code from GitHub...")
%cd /content/sensorspine-humaniser-v3
!git pull

# 3. Establish New Tunnel
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)

    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 BACKEND ACTIVE")
    print(f"COPY THIS URL TO FRONTEND: {public_url}")
    print("================================================\n")

    # 4. Start fresh (Redis + Celery + FastAPI)
    print("⚡ Launching clean backend pipeline...")
    %cd backend
    !chmod +x start.sh
    !./start.sh

except Exception as e:
    print(f"\n❌ FAILED TO START: {e}")
    print("Check your Colab Secrets for 'NGROK_TOKEN' and ensure 'Notebook access' is ON.")


Streaming output truncated to the last 5000 lines.
Loading weights:  18%|#8        | 62/339 [00:19<00:28,  9.77it/s]
INFO:     103.112.27.34:0 - "GET /status/b16f8ea7-fd33-45f2-9551-57fee9ccf431 HTTP/1.1" 200 OK
Loading weights:  79%|#######9  | 269/339 [00:54<00:13,  5.19it/s]
INFO:     115.242.248.226:0 - "GET /status/b16f8ea7-fd33-45f2-9551-57fee9ccf431 HTTP/1.1" 200 OK
Loading weights: 100%|##########| 339/339 [01:05<00:00,  5.18it/s]
[2026-05-11 20:19:49,686: INFO/ForkPoolWorker-1] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-11 20:19:49,706: INFO/ForkPoolWorker-1] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-7B-Instruct/f2826a00ceef68f0f2b946d945ecc0477ce4450c/generation_config.json "HTTP/1.1 200 OK"
[2026-05-11 20:19:49,796: INFO/ForkPoolWorker-1] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-7B-Instruct/resolve/main/custom_generate/generate.